# TSAD on Colab (GPU)

Adaptive Domain-Aware Representation Learning for Explainable TSAD — full E1–E6 runs.

**Setup:** set `REPO_URL` to your git remote below. No remote yet? Upload the repo as
`repo.zip` to `/content/` first (Files panel → Upload), the notebook will unzip it.
Results/checkpoints sync to Google Drive (`tsad_runs/`). Reruns skip finished experiments.

In [ ]:
import torch
print('cuda:', torch.cuda.is_available(), '| gpu:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)

from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/tsad_runs'
!mkdir -p {OUT}
print('drive out:', OUT)

In [ ]:
import os, shutil
REPO_URL = ''  # e.g. 'https://github.com/<you>/tsad.git' — leave empty to use /content/repo.zip
REPO = '/content/tsad'
if os.path.exists(REPO):
    shutil.rmtree(REPO)
if REPO_URL:
    !git clone {REPO_URL} {REPO}
else:
    assert os.path.exists('/content/repo.zip'), 'upload repo.zip or set REPO_URL'
    !unzip -q /content/repo.zip -d /content && mv /content/surya-1 {REPO}
%cd {REPO}
!pip install -q -r requirements.txt
import sys; assert sys.version_info >= (3, 10)

In [ ]:
# Smoke: fast pipeline proof on CPU/GPU (~3 min)
!python -m pytest -q 2>&1 | tail -2
!python scripts/data_summary.py --config configs/smoke.yaml 2>&1 | tail -1

In [ ]:
# Full runs E1–E6 (fast.yaml, seeds 0 1 2; use --seeds 0 1 2 3 4 for paper). Skips finished exps.
import subprocess
SEEDS = '0 1 2'
for exp in ['e1', 'e2', 'e3', 'e4', 'e5', 'e6', 'e7', 'e8']:
    done = [f for f in [f'reports/exp/{exp}.csv'] if __import__('os').path.exists(f)]
    if done:
        print(exp, 'exists -> skip (delete to rerun)'); continue
    print('== running', exp, '==', flush=True)
    subprocess.run(['python', 'scripts/run_experiment.py', '--config', 'configs/fast.yaml',
                      '--exp', exp, '--seeds'] + SEEDS.split() + ['--outdir', 'reports/exp'],
                     check=True)
    subprocess.run(['cp', '-r', 'reports', OUT + '/'], check=True)
    print('synced to Drive')

In [ ]:
# Gallery + bundle + adapt demo on GPU box
!python scripts/make_explanations.py --config configs/fast.yaml --opt seed=0 --out reports/explain_fast 2>&1 | tail -1
!python scripts/adapt.py --config configs/fast.yaml --epochs 5 --out reports/adapt_fast 2>&1 | tail -4
!python scripts/build_bundle.py --config configs/fast.yaml --name gpu_tcn 2>&1 | tail -1
!cp -r reports {OUT}/ && cp -r models {OUT}/ && echo synced

## Session timeouts & resume

- Colab kills idle sessions (~90 min) and long runs (~12 h). **Resume = rerun the E-loop cell**: finished
  experiments are skipped via their `reports/exp/<e>.csv` marker; Lightning `checkpoints/` resume:
  `Trainer(resume_from_checkpoint=...)` is wired in `train.py` via `ModelCheckpoint` (see `docs/GPU_RUNS.md`).
- Expected runtimes/memory: `docs/GPU_RUNS.md` (fast.yaml ≈ 30–60 min total on T4 for E1–E8, seeds 0–2).
- Paper runs: change `SEEDS` to `'0 1 2 3 4'` and `--config configs/default.yaml` (epochs 5).